# 📘 05_additional_metadata_tables.ipynb

This notebook extends the v0.1 Lakehouse Monitoring framework to include **custom metric metadata**.  
It introduces two new tables:

1. **`metric_templates`** — reusable definitions of metric rules  
2. **`metric_bindings`** — bindings that attach those templates to monitored tables  

Together with `monitors_control`, these three tables form the metadata-driven foundation of scalable Lakehouse Monitoring.

## 1️⃣ Widgets — Catalog and Schema Parameters

Define your working context (catalog, schemas, and asset directories).  
These widgets ensure this notebook can run across different workspaces without modification.

In [0]:
dbutils.widgets.text("catalog", "dbdemos_steventan", "Catalog")
dbutils.widgets.text("admin_schema", "monitoring_admin", "Admin Schema")
dbutils.widgets.text("out_schema", "lakehouse_monitoring_demo_results", "Output Schema")
dbutils.widgets.text("assets_dir_base", "/Workspace/Users/steven.tan@databricks.com/", "Assets Dir Base")
dbutils.widgets.text("data_schema", "lakehouse_monitoring", "Data Schema")

catalog = dbutils.widgets.get("catalog")
admin_schema = dbutils.widgets.get("admin_schema")
out_schema = dbutils.widgets.get("out_schema")
assets_dir_base = dbutils.widgets.get("assets_dir_base")
data_schema = dbutils.widgets.get("data_schema")

## 2️⃣ `metric_templates` — Define Reusable Metric Rules

Templates describe **how** to compute metrics — the logic, thresholds, and target dimension.

### Purpose
Each template defines a reusable rule such as:
- `missing_value_ratio`: check for NULLs  
- `negative_amount_ratio`: detect invalid negatives  
- `inconsistent_closed_claims_ratio`: enforce logical consistency  

### Schema Overview
| Column | Type | Description |
|---------|------|-------------|
| `template_name` | STRING | Unique metric identifier |
| `description` | STRING | Human-readable explanation |
| `metric_type` | STRING | Type (`AGGREGATE`, `DERIVED`, `DRIFT`) |
| `output_spark_type` | STRING | Output datatype |
| `input_columns` | ARRAY\<STRING> | Column references used in computation |
| `definition_template` | STRING | SQL/expr with `{{PARAM}}` placeholders |
| `default_params` | MAP\<STRING,STRING> | Default substitution values |
| `dimension` | STRING | Data quality dimension |
| `threshold_direction` | STRING | `LOWER_IS_BETTER` or `HIGHER_IS_BETTER` |
| `good_threshold` | DOUBLE | Threshold for good data |
| `acceptable_threshold` | DOUBLE | Threshold for acceptable data |

In [0]:
from pyspark.sql import Row, types as T

# Ensure schemas exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{admin_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{out_schema}")

TPL_TABLE = f"{catalog}.{admin_schema}.metric_templates"

ddl_templates = f"""
CREATE OR REPLACE TABLE {TPL_TABLE} (
  template_name       STRING NOT NULL,
  description         STRING,
  metric_type         STRING NOT NULL,
  output_spark_type   STRING NOT NULL,
  input_columns       ARRAY<STRING> NOT NULL,
  definition_template STRING NOT NULL,
  default_params      MAP<STRING,STRING>,
  dimension           STRING,
  threshold_direction STRING,
  good_threshold      DOUBLE,
  acceptable_threshold DOUBLE,
  PRIMARY KEY (template_name)
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'enabled')
"""
spark.sql(ddl_templates)
print(f"✅ Created table: {TPL_TABLE}")

### Insert Example Metric Templates

The examples below cover multiple data quality dimensions:
- **Completeness**: Missing or NULL values  
- **Validity**: Negative or invalid numbers  
- **Consistency**: Cross-field checks  
- **Accuracy**: Out-of-range values

In [0]:
FUNC_SCHEMA = f"{catalog}.{admin_schema}"

templates = [
    # ─────────────────────────────
    # COMPLETENESS
    # ─────────────────────────────
    Row(
        template_name="missing_value_ratio",
        description="Ratio of records with NULL or empty values in the specified column",
        metric_type="AGGREGATE",
        output_spark_type="double",
        input_columns=[":table"],
        definition_template=f"avg({FUNC_SCHEMA}.rule_missing_value_ratio_bit({{{{VAL_COL}}}}))",
        default_params={"REASON": "missing_value"},
        dimension="COMPLETENESS",
        threshold_direction="LOWER_IS_BETTER",
        good_threshold=0.01,
        acceptable_threshold=0.05
    ),
    Row(
        template_name="missing_value_details_json",
        description="Captures detailed information about records with missing or empty values",
        metric_type="AGGREGATE",
        output_spark_type="string",
        input_columns=[":table"],
        definition_template=(
            "to_json(coalesce("
            f"  collect_list(to_json({FUNC_SCHEMA}.rule_missing_value_details(CAST({{{{PK}}}} AS STRING), '{{{{COLNAME}}}}', {{{{VAL_COL}}}}))),"
            "  CAST(array() AS array<string>)"
            "))"
        ),
        default_params={"REASON": "missing_value"},
        dimension=None,
        threshold_direction=None,
        good_threshold=None,
        acceptable_threshold=None
    ),

    # ─────────────────────────────
    # VALIDITY
    # ─────────────────────────────
    Row(
        template_name="end_before_start_ratio",
        description="Ratio of records where end date occurs before start date",
        metric_type="AGGREGATE",
        output_spark_type="double",
        input_columns=[":table"],
        definition_template=f"avg({FUNC_SCHEMA}.rule_end_before_start_ratio_bit({{{{START_DATE}}}}, {{{{END_DATE}}}}))",
        default_params={"REASON": "end_before_start"},
        dimension="VALIDITY",
        threshold_direction="LOWER_IS_BETTER",
        good_threshold=0.01,
        acceptable_threshold=0.05
    ),
    Row(
        template_name="end_before_start_details_json",
        description="Captures detailed information about records where end date is before start date",
        metric_type="AGGREGATE",
        output_spark_type="string",
        input_columns=[":table"],
        definition_template=(
            "to_json(coalesce("
            f"  collect_list(to_json({FUNC_SCHEMA}.rule_end_before_start_details(CAST({{{{PK}}}} AS STRING), {{{{START_DATE}}}}, {{{{END_DATE}}}}))),"
            "  CAST(array() AS array<string>)"
            "))"
        ),
        default_params={"REASON": "end_before_start"},
        dimension=None,
        threshold_direction=None,
        good_threshold=None,
        acceptable_threshold=None
    ),
    
    Row(
        template_name="future_timestamp_ratio",
        description="Ratio of records with timestamps in the future",
        metric_type="AGGREGATE",
        output_spark_type="double",
        input_columns=[":table"],
        definition_template=f"avg({FUNC_SCHEMA}.rule_future_timestamp_ratio_bit({{{{TS_COL}}}}))",
        default_params={"REASON": "future_timestamp"},
        dimension="VALIDITY",
        threshold_direction="LOWER_IS_BETTER",
        good_threshold=0.01,
        acceptable_threshold=0.05
    ),
    Row(
        template_name="future_timestamp_details_json",
        description="Captures detailed information about records with future timestamps",
        metric_type="AGGREGATE",
        output_spark_type="string",
        input_columns=[":table"],
        definition_template=(
            "to_json(coalesce("
            f"  collect_list(to_json({FUNC_SCHEMA}.rule_future_timestamp_details(CAST({{{{PK}}}} AS STRING), {{{{TS_COL}}}}))),"
            "  CAST(array() AS array<string>)"
            "))"
        ),
        default_params={"REASON": "future_timestamp"},
        dimension=None,
        threshold_direction=None,
        good_threshold=None,
        acceptable_threshold=None
    ),

    Row(
        template_name="negative_amount_ratio",
        description="Ratio of records where numeric amount is negative (< 0)",
        metric_type="AGGREGATE",
        output_spark_type="double",
        input_columns=[":table"],
        definition_template=f"avg({FUNC_SCHEMA}.rule_negative_amount_ratio_bit({{{{AMOUNT_COL}}}}))",
        default_params={"REASON": "negative_amount"},
        dimension="VALIDITY",
        threshold_direction="LOWER_IS_BETTER",
        good_threshold=0.01,
        acceptable_threshold=0.05
    ),
    Row(
        template_name="negative_amount_details_json",
        description="Captures detailed information about records with negative amounts",
        metric_type="AGGREGATE",
        output_spark_type="string",
        input_columns=[":table"],
        definition_template=(
            "to_json(coalesce("
            f"  collect_list(to_json({FUNC_SCHEMA}.rule_negative_amount_details(CAST({{{{PK}}}} AS STRING), {{{{AMOUNT_COL}}}}))),"
            "  CAST(array() AS array<string>)"
            "))"
        ),
        default_params={"REASON": "negative_amount"},
        dimension=None,
        threshold_direction=None,
        good_threshold=None,
        acceptable_threshold=None
    ),

    Row(
        template_name="invalid_date_string_ratio",
        description="Ratio of records with strings that cannot be parsed as valid dates",
        metric_type="AGGREGATE",
        output_spark_type="double",
        input_columns=[":table"],
        definition_template=f"avg({FUNC_SCHEMA}.rule_invalid_date_string_ratio_bit({{{{DATE_STR}}}}, {{{{FMT}}}}))",
        default_params={"REASON": "invalid_date_string"},
        dimension="VALIDITY",
        threshold_direction="LOWER_IS_BETTER",
        good_threshold=0.01,
        acceptable_threshold=0.05
    ),
    Row(
        template_name="invalid_date_string_details_json",
        description="Captures detailed information about malformed date strings",
        metric_type="AGGREGATE",
        output_spark_type="string",
        input_columns=[":table"],
        definition_template=(
            "to_json(coalesce("
            f"  collect_list(to_json({FUNC_SCHEMA}.rule_invalid_date_string_details(CAST({{{{PK}}}} AS STRING), {{{{DATE_STR}}}}, {{{{FMT}}}}))),"
            "  CAST(array() AS array<string>)"
            "))"
        ),
        default_params={"REASON": "invalid_date_string"},
        dimension=None,
        threshold_direction=None,
        good_threshold=None,
        acceptable_threshold=None
    ),

    Row(
        template_name="unexpected_category_ratio",
        description="Ratio of records with category values not in the expected set",
        metric_type="AGGREGATE",
        output_spark_type="double",
        input_columns=[":table"],
        definition_template=f"avg({FUNC_SCHEMA}.rule_unexpected_category_ratio_bit({{{{VAL}}}}, {{{{CSV_EXPECTED}}}}))",
        default_params={"REASON": "unexpected_category"},
        dimension="VALIDITY",
        threshold_direction="LOWER_IS_BETTER",
        good_threshold=0.01,
        acceptable_threshold=0.05
    ),
    Row(
        template_name="unexpected_category_details_json",
        description="Captures detailed information about unexpected category values",
        metric_type="AGGREGATE",
        output_spark_type="string",
        input_columns=[":table"],
        definition_template=(
            "to_json(coalesce("
            f"  collect_list(to_json({FUNC_SCHEMA}.rule_unexpected_category_details(CAST({{{{PK}}}} AS STRING), {{{{VAL}}}}, {{{{CSV_EXPECTED}}}}))),"
            "  CAST(array() AS array<string>)"
            "))"
        ),
        default_params={"REASON": "unexpected_category"},
        dimension=None,
        threshold_direction=None,
        good_threshold=None,
        acceptable_threshold=None
    ),

    # ─────────────────────────────
    # CONSISTENCY
    # ─────────────────────────────
    Row(
        template_name="inconsistent_closed_claims_ratio",
        description="Ratio of records where status indicates closure but timestamp is missing",
        metric_type="AGGREGATE",
        output_spark_type="double",
        input_columns=[":table"],
        definition_template=f"avg({FUNC_SCHEMA}.rule_inconsistent_closed_claims_ratio_bit({{{{STATUS_COL}}}}, {{{{CLOSED_AT_COL}}}}))",
        default_params={"REASON": "inconsistent_closed_claims"},
        dimension="CONSISTENCY",
        threshold_direction="LOWER_IS_BETTER",
        good_threshold=0.02,
        acceptable_threshold=0.10
    ),
    Row(
        template_name="inconsistent_closed_claims_details_json",
        description="Captures detailed information about records with inconsistent status and timestamp",
        metric_type="AGGREGATE",
        output_spark_type="string",
        input_columns=[":table"],
        definition_template=(
            "to_json(coalesce("
            f"  collect_list(to_json({FUNC_SCHEMA}.rule_inconsistent_closed_claims_details(CAST({{{{PK}}}} AS STRING), {{{{STATUS_COL}}}}, {{{{CLOSED_AT_COL}}}}))),"
            "  CAST(array() AS array<string>)"
            "))"
        ),
        default_params={"REASON": "inconsistent_closed_claims"},
        dimension=None,
        threshold_direction=None,
        good_threshold=None,
        acceptable_threshold=None
    ),

    # ─────────────────────────────
    # ACCURACY
    # ─────────────────────────────
    Row(
        template_name="premium_out_of_range_ratio",
        description="Ratio of records where amount value is outside expected range",
        metric_type="AGGREGATE",
        output_spark_type="double",
        input_columns=[":table"],
        definition_template=f"avg({FUNC_SCHEMA}.rule_premium_out_of_range_ratio_bit({{{{AMOUNT_COL}}}}))",
        default_params={"REASON": "premium_out_of_range"},
        dimension="ACCURACY",
        threshold_direction="LOWER_IS_BETTER",
        good_threshold=0.01,
        acceptable_threshold=0.05
    ),
    Row(
        template_name="premium_out_of_range_details_json",
        description="Captures detailed information about records with out-of-range amount values",
        metric_type="AGGREGATE",
        output_spark_type="string",
        input_columns=[":table"],
        definition_template=(
            "to_json(coalesce("
            f"  collect_list(to_json({FUNC_SCHEMA}.rule_premium_out_of_range_details(CAST({{{{PK}}}} AS STRING), {{{{AMOUNT_COL}}}}))),"
            "  CAST(array() AS array<string>)"
            "))"
        ),
        default_params={"REASON": "premium_out_of_range"},
        dimension=None,
        threshold_direction=None,
        good_threshold=None,
        acceptable_threshold=None
    ),

    Row(
        template_name="age_out_of_range_ratio",
        description="Ratio of records where age value is outside reasonable range",
        metric_type="AGGREGATE",
        output_spark_type="double",
        input_columns=[":table"],
        definition_template=f"avg({FUNC_SCHEMA}.rule_age_out_of_range_ratio_bit({{{{AGE}}}}, {{{{MIN_AGE}}}}, {{{{MAX_AGE}}}}))",
        default_params={"REASON": "age_out_of_range"},
        dimension="ACCURACY",
        threshold_direction="LOWER_IS_BETTER",
        good_threshold=0.01,
        acceptable_threshold=0.05
    ),
    Row(
        template_name="age_out_of_range_details_json",
        description="Captures detailed information about records with out-of-range ages",
        metric_type="AGGREGATE",
        output_spark_type="string",
        input_columns=[":table"],
        definition_template=(
            "to_json(coalesce("
            f"  collect_list(to_json({FUNC_SCHEMA}.rule_age_out_of_range_details(CAST({{{{PK}}}} AS STRING), {{{{AGE}}}}, {{{{MIN_AGE}}}}, {{{{MAX_AGE}}}}))),"
            "  CAST(array() AS array<string>)"
            "))"
        ),
        default_params={"REASON": "age_out_of_range"},
        dimension=None,
        threshold_direction=None,
        good_threshold=None,
        acceptable_threshold=None
    ),

    Row(
        template_name="outlier_ratio",
        description="Ratio of records with values outside statistical bounds (outliers)",
        metric_type="AGGREGATE",
        output_spark_type="double",
        input_columns=[":table"],
        definition_template=f"avg({FUNC_SCHEMA}.rule_outlier_ratio_bit({{{{VAL}}}}, {{{{LOWER_BOUND}}}}, {{{{UPPER_BOUND}}}}))",
        default_params={"REASON": "statistical_outlier"},
        dimension="ACCURACY",
        threshold_direction="LOWER_IS_BETTER",
        good_threshold=0.05,
        acceptable_threshold=0.10
    ),
    Row(
        template_name="outlier_details_json",
        description="Captures detailed information about statistical outliers",
        metric_type="AGGREGATE",
        output_spark_type="string",
        input_columns=[":table"],
        definition_template=(
            "to_json(coalesce("
            f"  collect_list(to_json({FUNC_SCHEMA}.rule_outlier_details(CAST({{{{PK}}}} AS STRING), {{{{VAL}}}}, {{{{LOWER_BOUND}}}}, {{{{UPPER_BOUND}}}}))),"
            "  CAST(array() AS array<string>)"
            "))"
        ),
        default_params={"REASON": "statistical_outlier"},
        dimension=None,
        threshold_direction=None,
        good_threshold=None,
        acceptable_threshold=None
    ),

    Row(
        template_name="invalid_percentage_ratio",
        description="Ratio of records where percentage value is outside valid range (0-100 or 0-1)",
        metric_type="AGGREGATE",
        output_spark_type="double",
        input_columns=[":table"],
        definition_template=f"avg({FUNC_SCHEMA}.rule_invalid_percentage_ratio_bit({{{{PCT}}}}, {{{{IS_DECIMAL}}}}))",
        default_params={"REASON": "percentage_out_of_range"},
        dimension="ACCURACY",
        threshold_direction="LOWER_IS_BETTER",
        good_threshold=0.01,
        acceptable_threshold=0.05
    ),
    Row(
        template_name="invalid_percentage_details_json",
        description="Captures detailed information about invalid percentages",
        metric_type="AGGREGATE",
        output_spark_type="string",
        input_columns=[":table"],
        definition_template=(
            "to_json(coalesce("
            f"  collect_list(to_json({FUNC_SCHEMA}.rule_invalid_percentage_details(CAST({{{{PK}}}} AS STRING), {{{{PCT}}}}, {{{{IS_DECIMAL}}}}))),"
            "  CAST(array() AS array<string>)"
            "))"
        ),
        default_params={"REASON": "percentage_out_of_range"},
        dimension=None,
        threshold_direction=None,
        good_threshold=None,
        acceptable_threshold=None
    ),
]

schema = T.StructType([
    T.StructField("template_name",       T.StringType(), False),
    T.StructField("description",         T.StringType(), True),
    T.StructField("metric_type",         T.StringType(), False),
    T.StructField("output_spark_type",   T.StringType(), False),
    T.StructField("input_columns",       T.ArrayType(T.StringType()), False),
    T.StructField("definition_template", T.StringType(), False),
    T.StructField("default_params",      T.MapType(T.StringType(), T.StringType()), True),
    T.StructField("dimension",           T.StringType(), True),
    T.StructField("threshold_direction", T.StringType(), True),
    T.StructField("good_threshold",      T.DoubleType(), True),
    T.StructField("acceptable_threshold",T.DoubleType(), True),
])

spark.createDataFrame(templates, schema).write.mode("overwrite").saveAsTable(TPL_TABLE)
print(f"✅ Inserted {len(templates)} templates into {TPL_TABLE}")

## 3️⃣ `metric_bindings` — Bind Templates to Tables

Bindings link templates to actual tables from `monitors_control`.

Each record defines:
- Which table the rule applies to  
- The metric name (instance of a template)  
- Parameter substitutions (column names)  
- Whether the rule is enabled  

### Schema Overview
| Column | Type | Description |
|---------|------|-------------|
| `table_catalog` | STRING | Catalog of monitored table |
| `table_schema` | STRING | Schema of monitored table |
| `table_name` | STRING | Table name |
| `metric_name` | STRING | Unique metric identifier |
| `template_name` | STRING | Reference to `metric_templates.template_name` |
| `metric_type` | STRING | Metric type (`AGGREGATE`, etc.) |
| `output_spark_type` | STRING | Spark datatype |
| `input_columns` | ARRAY\<STRING> | Referenced columns |
| `params` | MAP\<STRING,STRING> | Parameter substitutions |
| `enabled` | BOOLEAN | Enable/disable metric |

In [0]:
BIND_TABLE = f"{catalog}.{admin_schema}.metric_bindings"

ddl_bindings = f"""
CREATE OR REPLACE TABLE {BIND_TABLE} (
  table_catalog      STRING NOT NULL,
  table_schema       STRING NOT NULL,
  table_name         STRING NOT NULL,
  metric_name        STRING NOT NULL,
  template_name      STRING NOT NULL,
  metric_type        STRING NOT NULL,
  output_spark_type  STRING NOT NULL,
  input_columns      ARRAY<STRING> NOT NULL,
  params             MAP<STRING,STRING>,
  enabled            BOOLEAN DEFAULT true
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'enabled')
"""
spark.sql(ddl_bindings)
print(f"✅ Created table: {BIND_TABLE}")

### Insert Example Metric Bindings

This maps templates to specific tables (e.g., `claims`, `premium_billing`, `policies`).  
Each record corresponds to one metric configuration that the API will later register automatically.

In [0]:
bindings = [
    # ───────── claims ─────────
    Row(  # VALIDITY: negative amount
        table_catalog=catalog, table_schema=data_schema, table_name="claims",
        metric_name="negative_amount_ratio",
        template_name="negative_amount_ratio",
        metric_type="AGGREGATE", output_spark_type="double",
        input_columns=[":table"],
        params={"AMOUNT_COL": "claim_amount"},
        enabled=True
    ),
    Row(  # details
        table_catalog=catalog, table_schema=data_schema, table_name="claims",
        metric_name="negative_amount_details_json",
        template_name="negative_amount_details_json",
        metric_type="AGGREGATE", output_spark_type="string",
        input_columns=[":table"],
        params={"AMOUNT_COL": "claim_amount", "PK": "claim_id"},
        enabled=True
    ),

    Row(  # CONSISTENCY: CLOSED must have closed_at
        table_catalog=catalog, table_schema=data_schema, table_name="claims",
        metric_name="inconsistent_closed_claims_ratio",
        template_name="inconsistent_closed_claims_ratio",
        metric_type="AGGREGATE", output_spark_type="double",
        input_columns=[":table"],
        params={"STATUS_COL": "claim_status", "CLOSED_AT_COL": "closed_at"},
        enabled=True
    ),
    Row(  # details
        table_catalog=catalog, table_schema=data_schema, table_name="claims",
        metric_name="inconsistent_closed_claims_details_json",
        template_name="inconsistent_closed_claims_details_json",
        metric_type="AGGREGATE", output_spark_type="string",
        input_columns=[":table"],
        params={"STATUS_COL": "claim_status", "CLOSED_AT_COL": "closed_at", "PK": "claim_id"},
        enabled=True
    ),

    # ───────── premium_billing ─────────
    Row(  # ACCURACY: out-of-range premium
        table_catalog=catalog, table_schema=data_schema, table_name="premium_billing",
        metric_name="premium_out_of_range_ratio",
        template_name="premium_out_of_range_ratio",
        metric_type="AGGREGATE", output_spark_type="double",
        input_columns=[":table"],
        params={"AMOUNT_COL": "amount_due"},
        enabled=True
    ),
    Row(  # details
        table_catalog=catalog, table_schema=data_schema, table_name="premium_billing",
        metric_name="premium_out_of_range_details_json",
        template_name="premium_out_of_range_details_json",
        metric_type="AGGREGATE", output_spark_type="string",
        input_columns=[":table"],
        params={"AMOUNT_COL": "amount_due", "PK": "invoice_id"},
        enabled=True
    ),

    # ───────── policies ─────────
    Row(  # COMPLETENESS: missing value
        table_catalog=catalog, table_schema=data_schema, table_name="policies",
        metric_name="missing_value_ratio",
        template_name="missing_value_ratio",
        metric_type="AGGREGATE", output_spark_type="double",
        input_columns=[":table"],
        params={"VAL_COL": "policy_no"},
        enabled=True
    ),
    Row(  # details
        table_catalog=catalog, table_schema=data_schema, table_name="policies",
        metric_name="missing_value_details_json",
        template_name="missing_value_details_json",
        metric_type="AGGREGATE", output_spark_type="string",
        input_columns=[":table"],
        params={"VAL_COL": "policy_no", "COLNAME": "policy_no", "PK": "policy_no"},
        enabled=True
    ),
]

schema_bind = T.StructType([
    T.StructField("table_catalog",     T.StringType(), False),
    T.StructField("table_schema",      T.StringType(), False),
    T.StructField("table_name",        T.StringType(), False),
    T.StructField("metric_name",       T.StringType(), False),
    T.StructField("template_name",     T.StringType(), False),
    T.StructField("metric_type",       T.StringType(), False),
    T.StructField("output_spark_type", T.StringType(), False),
    T.StructField("input_columns",     T.ArrayType(T.StringType()), False),
    T.StructField("params",            T.MapType(T.StringType(), T.StringType()), True),
    T.StructField("enabled",           T.BooleanType(), True),
])

spark.createDataFrame(bindings, schema_bind).write.mode("overwrite").saveAsTable(BIND_TABLE)
print(f"✅ Inserted {len(bindings)} bindings into {BIND_TABLE}")

## ✅ Summary

You now have three metadata tables forming the backbone of metadata-driven Lakehouse Monitoring:

| Table | Purpose |
|--------|----------|
| `monitors_control` | Defines *what* to monitor (tables, schedule, assets) |
| `metric_templates` | Defines *how* to monitor (logic, thresholds, dimension) |
| `metric_bindings` | Defines *which* metric applies to *which* table |

Next, you’ll use these metadata tables in the Lakehouse Monitoring API notebook to automatically register and refresh monitors.